In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data_estaciones")
NDVI_PATH = Path("ndvi_semanal_estaciones_2021_2025.csv")

# Columnas que identifican la fila, no atributos a agregar
KEY_COLS = {"date", "semana", "anio", "estacion"}

# Prefijo de las dummies one-hot de direccion del viento
ONEHOT_PREFIX = "WDR_"


def to_monday(s: pd.Series) -> pd.Series:
    """Lunes de la semana de cada fecha.

    Nota: NO usar dt.to_period("W-MON"), porque en pandas "W-MON" significa
    semana *terminada* en lunes, asi que start_time cae en martes y no empata
    con la columna 'semana' del NDVI (que si son lunes).
    """
    s = pd.to_datetime(s, errors="coerce")
    return (s - pd.to_timedelta(s.dt.weekday, unit="D")).dt.normalize()


def build_weekly_station(df: pd.DataFrame, station: str) -> pd.DataFrame:
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d.dropna(subset=["date"]).copy()

    d["semana"] = to_monday(d["date"])

    numeric_cols = [
        col for col in d.columns
        if col not in KEY_COLS and pd.api.types.is_numeric_dtype(d[col])
    ]
    # Las dummies 0/1 van por media (= proporcion de horas de la semana con
    # esa direccion de viento); la mediana de una dummy solo daria 0 o 1.
    onehot_cols = [c for c in numeric_cols if c.startswith(ONEHOT_PREFIX)]
    median_cols = [c for c in numeric_cols if c not in onehot_cols]

    agg = {c: "median" for c in median_cols}
    agg.update({c: "mean" for c in onehot_cols})

    weekly = (
        d.groupby("semana", as_index=False)
        .agg(agg)
        .sort_values("semana")
        .reset_index(drop=True)
    )

    weekly["estacion"] = station
    weekly["anio"] = weekly["semana"].dt.year
    return weekly[["estacion", "anio", "semana"] + numeric_cols]


def main():
    ndvi = pd.read_csv(NDVI_PATH)
    ndvi["semana"] = to_monday(ndvi["semana"])
    ndvi = ndvi.dropna(subset=["semana"]).copy()

    sources = [
        ("NE2", DATA_DIR / "BD_NE2_limpia.csv"),
        ("NE3", DATA_DIR / "BD_NE3_limpia.csv"),
    ]

    for station_name, src_path in sources:
        weekly = build_weekly_station(pd.read_csv(src_path), station_name)
        ndvi_station = (
            ndvi.loc[ndvi["estacion"] == station_name, ["semana", "ndvi_mediana"]]
            .drop_duplicates(subset="semana")
        )

        # outer: conserva las semanas de la estacion Y las del NDVI
        out_all = (
            weekly.merge(ndvi_station, on="semana", how="outer")
            .sort_values("semana")
            .reset_index(drop=True)
        )
        # Rellenar llaves en semanas que solo existian en el NDVI
        out_all["estacion"] = station_name
        out_all["anio"] = out_all["semana"].dt.year

        # "Sin nulos" = sin semanas sin NDVI; se conservan nulos aislados
        # en contaminantes (PM2.5 falta en casi toda la serie de NE3).
        out_no_null = out_all[out_all["ndvi_mediana"].notna()].reset_index(drop=True)

        out_all_path = DATA_DIR / f"{station_name}_semanal_mediana_con_ndvi.csv"
        out_all.to_csv(out_all_path, index=False)

        out_no_null_path = DATA_DIR / f"{station_name}_semanal_mediana_con_ndvi_sin_nulos.csv"
        out_no_null.to_csv(out_no_null_path, index=False)

        print(f"Generado: {out_all_path.name} ({len(out_all)} filas)")
        print(f"Generado: {out_no_null_path.name} ({len(out_no_null)} filas)")


if __name__ == "__main__":
    main()